# Nocturne Aegis Cel Candidate Generator v4

This is the **a Colab-first iteration notebook** for `Nocturne Aegis Cel`.

Its purpose is to generate a **small, high-signal first pass of 3 images** for `Nocturne Aegis Cel`, to improve the base prompt to generate training candidates.

**Changelog:**

- `v1` preserved more of the style language, but the prose prompt was too long and got truncated by CLIP.
- `v2` fixed several SDXL issues, but leaned too far toward generic tag cleanup and still risked weak style steering.
- `v3` protected token length, but became too light on notebook guidance and silently trimmed prompts, which can hide failures.

This notebook balances those tradeoffs:

- clear Colab notes and documentation
- `StableDiffusionXLPipeline` with SDXL prompt channels
- compact style bundles that fit token limits
- explicit token reporting before generation
- **fail-fast** prompt validation instead of silent trimming by default
- a small default run: **3 prompts x 1 seed = 3 images**

After the 3-image pass looks acceptable, this notebook can be scaled into broader candidate generation for LoRA dataset curation.


## Working Principle

The base model does **not** know `Nocturne Aegis Cel` yet. Before the LoRA exists, the style must be expressed as **mechanical visual instructions** the base checkpoint can understand:

- 1990s cel-animation structure
- modern volumetric finish
- sharp geometric face design
- glass-like eyes
- jewel-toned palette
- layered cel shadows
- ambient occlusion
- iridescent metal and matte cloth separation

That style signal has to be strong enough to matter, but short enough to survive CLIP. The notebook therefore splits prompting across:

- `prompt`: quality tags + content tags + short style anchor
- `prompt_2`: compact global rendering/style rules
- `negative_prompt` and `negative_prompt_2`: explicit blockers for sketch, monochrome, unfinished output, anatomy errors, and clutter


In [ ]:
!pip install -q -U diffusers transformers accelerate safetensors pillow pandas tqdm huggingface_hub

In [ ]:
import gc
import inspect
import json
import zipfile
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

from diffusers import EulerAncestralDiscreteScheduler, StableDiffusionXLPipeline

## Optional Colab Setup

Use these only when needed.

- Mount Google Drive if you want outputs to persist across runtime resets.
- Log in to Hugging Face only if model access fails.


In [ ]:
# Optional: mount Google Drive for persistent storage.
# from google.colab import drive
# drive.mount('/content/drive')

# Optional: Hugging Face login if model download requires authentication.
# from huggingface_hub import notebook_login
# notebook_login()

## Configuration

This notebook defaults to a **3-image iteration run** so you can inspect prompt quality before spending time on a larger batch.

Recommended first pass:

- `NUM_IMAGES_PER_PROMPT = 1`
- `NUM_INFERENCE_STEPS = 38`
- `GUIDANCE_SCALE = 7.0`
- `CLIP_SKIP = 2`
- `USE_CPU_OFFLOAD = True` on T4

Only scale up after the smoke run looks clearly polished and on-style.


In [ ]:
# =========================
# CONFIG
# =========================

MODEL_ID = "OnomaAIResearch/Illustrious-xl-early-release-v0"
PROMPTS_FILE = Path("src/prompts_illustrious_v4_iteration.json")

# Change this to a Drive path if you mounted Drive.
OUTPUT_ROOT = Path("outputs/nocturne_aegis_candidates_v4")
IMAGES_DIR = OUTPUT_ROOT / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# Small first pass: 3 prompts x 1 image each.
NUM_IMAGES_PER_PROMPT = 1
BASE_SEED = 250505

NUM_INFERENCE_STEPS = 38
GUIDANCE_SCALE = 7.0
GUIDANCE_RESCALE = 0.0
CLIP_SKIP = 2

# T4: True. L4 / A100: False is usually faster.
USE_CPU_OFFLOAD = True
SCHEDULER = "euler_a"  # "euler_a" or "default"

# Prompt validation behavior.
MAX_TOKENS_PER_CHANNEL = 70
FAIL_ON_TOKEN_OVERFLOW = True

ASPECT_SIZES = {
    "portrait": (896, 1152),
    "full_body": (832, 1216),
    "wide": (1216, 832),
    "square": (1024, 1024),
}

print("Output root:", OUTPUT_ROOT)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Prompt Set

The default prompt file contains **3 deliberately different subjects**:

1. portrait / face-read test
2. full-body figure / silhouette test
3. mech-environment / hard-surface test

This is more useful than generating 3 near-duplicate characters. If these three images fail, scaling to 80 images will only scale the failure.


In [ ]:
if not PROMPTS_FILE.exists():
    raise FileNotFoundError(f"Prompt file not found: {PROMPTS_FILE}")

with open(PROMPTS_FILE, "r", encoding="utf-8") as f:
    prompt_rows = json.load(f)

assert len(prompt_rows) == 3, f"Expected 3 iteration prompts, found {len(prompt_rows)}"
for row in prompt_rows:
    assert row["caption"].startswith("nacel_v1,"), f"Caption must start with nacel_v1: {row['id']}"

pd.DataFrame(prompt_rows)

## Prompt Architecture

The style language here is intentionally **compact but specific**.

Why this structure:

- We keep the content prompt short and concrete.
- We avoid the long prose block from `v1` that overflowed CLIP.
- We avoid over-thinning the style signal the way `v3` risked doing.
- We keep the strongest rendering controls in `prompt_2`, where SDXL can still use them without overloading the main prompt.

This is the balance point we want to iterate from in Colab.


In [ ]:
QUALITY_TAGS = (
    "masterpiece, best quality, amazing quality, very aesthetic, "
    "finished full-color anime illustration, polished final render, sharp focus"
)

STYLE_ANCHOR = (
    "Nocturne Aegis Cel, 90s cel-animation structure, modern volumetric finish, "
    "jewel-tone palette, glass-like eyes, dramatic rim light"
)

PROMPT_2_STYLE = (
    "elongated elegant proportions, sharp geometric face planes, melancholic expression, "
    "tapered linework, bold hard-surface contours, layered cel shadows, ambient occlusion, "
    "dramatic chiaroscuro, iridescent metal, matte cloth, clear silhouette, controlled detail hierarchy"
)

NEGATIVE_PROMPT = (
    "worst quality, low quality, lowres, sketch, rough sketch, pencil drawing, monochrome, grayscale, "
    "lineart only, unfinished, draft, blurry, muddy colors, flat lighting, text, watermark, logo, signature, "
    "bad anatomy, bad hands, extra fingers, missing fingers, malformed limbs, cropped hands, deformed face, plastic skin"
)

NEGATIVE_PROMPT_2 = (
    "sketch, monochrome, grayscale, unfinished, noisy, blurry, generic anime drift, photorealistic skin, "
    "bad anatomy, bad hands, extra fingers, text, watermark, logo"
)


def build_positive_prompt(content_tags: str) -> str:
    return f"{QUALITY_TAGS}, {content_tags}, {STYLE_ANCHOR}

## Load Model

The notebook uses `StableDiffusionXLPipeline`, not the generic `DiffusionPipeline` from `v1`.

Important choices:

- `clip_skip=2` is kept configurable because anime checkpoints often respond better to it.
- `Euler A` remains the default scheduler because it is a reasonable first choice for stylized SDXL sampling.
- CPU offload stays enabled by default for T4 compatibility.


In [ ]:
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    use_safetensors=True,
)

if SCHEDULER == "euler_a":
    pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)

pipe.enable_vae_slicing()
try:
    pipe.enable_vae_tiling()
except Exception:
    pass

if torch.cuda.is_available():
    if USE_CPU_OFFLOAD:
        pipe.enable_model_cpu_offload()
        print("Model CPU offload enabled.")
    else:
        pipe.to("cuda")
        print("Pipeline moved to CUDA.")
else:
    print("CUDA unavailable. Running on CPU will be very slow.")

## Token Validation

This notebook does **not** silently trim prompts by default.

That is intentional. Silent trimming can hide the exact reason a run drifts off-style. Instead, this cell reports token counts clearly and fails if a prompt channel is too long.

If you later decide you want auto-trimming, add it deliberately after you know which tags matter most.


In [ ]:
def token_count(text, tokenizer):
    ids = tokenizer(text, truncation=False, add_special_tokens=True).input_ids
    return len(ids)


def prompt_report(prompt, prompt_2, neg, neg_2):
    tok1 = pipe.tokenizer
    tok2 = pipe.tokenizer_2 if getattr(pipe, "tokenizer_2", None) is not None else pipe.tokenizer

    report = {
        "prompt_tokens": token_count(prompt, tok1),
        "prompt_2_tokens": token_count(prompt_2, tok2),
        "negative_prompt_tokens": token_count(neg, tok1),
        "negative_prompt_2_tokens": token_count(neg_2, tok2),
    }
    return report


def validate_prompt_channels(prompt, prompt_2, neg, neg_2, max_tokens=MAX_TOKENS_PER_CHANNEL):
    report = prompt_report(prompt, prompt_2, neg, neg_2)
    print(report)
    too_long = {k: v for k, v in report.items() if v > max_tokens}
    if too_long and FAIL_ON_TOKEN_OVERFLOW:
        raise ValueError(f"Prompt channel exceeds {max_tokens} tokens: {too_long}")
    return report


sample_positive = build_positive_prompt(prompt_rows[0]["content_tags"])
sample_report = validate_prompt_channels(
    sample_positive,
    PROMPT_2_STYLE,
    NEGATIVE_PROMPT,
    NEGATIVE_PROMPT_2,
)

print("\nSample prompt:
", sample_positive)
print("\nPrompt 2:
", PROMPT_2_STYLE)

## Generation Helpers

`call_pipe()` sends both SDXL prompt channels and checks whether `clip_skip` and `guidance_rescale` are supported by the installed pipeline version.


In [ ]:
def call_pipe(prompt, prompt_2, negative_prompt, negative_prompt_2, width, height, seed):
    generator = torch.Generator(device="cuda" if torch.cuda.is_available() else "cpu").manual_seed(seed)

    kwargs = dict(
        prompt=prompt,
        prompt_2=prompt_2,
        negative_prompt=negative_prompt,
        negative_prompt_2=negative_prompt_2,
        width=width,
        height=height,
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
        original_size=(width, height),
        target_size=(width, height),
        crops_coords_top_left=(0, 0),
    )

    sig = inspect.signature(pipe.__call__).parameters
    if "clip_skip" in sig and CLIP_SKIP is not None:
        kwargs["clip_skip"] = CLIP_SKIP
    if "guidance_rescale" in sig:
        kwargs["guidance_rescale"] = GUIDANCE_RESCALE

    with torch.inference_mode():
        return pipe(**kwargs).images[0]


def make_contact_sheet(image_paths, out_path, thumb_w=240, thumb_h=240, cols=3):
    image_paths = [Path(p) for p in image_paths]
    if not image_paths:
        return None

    rows = (len(image_paths) + cols - 1) // cols
    label_h = 34
    sheet = Image.new("RGB", (cols * thumb_w, rows * (thumb_h + label_h)), "white")
    draw = ImageDraw.Draw(sheet)

    for i, path in enumerate(image_paths):
        img = Image.open(path).convert("RGB")
        img.thumbnail((thumb_w, thumb_h))
        x = (i % cols) * thumb_w + (thumb_w - img.width) // 2
        y = (i // cols) * (thumb_h + label_h)
        sheet.paste(img, (x, y))
        draw.text((i % cols * thumb_w + 6, y + thumb_h + 6), path.stem[:32], fill=(0, 0, 0))

    sheet.save(out_path, quality=92)
    return out_path

## Preflight Check

This is the most important inspection cell before generation.

Review:

- token counts
- actual text being sent to the model
- chosen resolution per prompt

If these inputs are wrong, the output will be wrong. Do not skip this cell.


In [ ]:
preflight_rows = []
for row in prompt_rows:
    aspect = row.get("aspect", "square")
    width, height = ASPECT_SIZES.get(aspect, ASPECT_SIZES["square"])
    positive = build_positive_prompt(row["content_tags"])
    report = validate_prompt_channels(positive, PROMPT_2_STYLE, NEGATIVE_PROMPT, NEGATIVE_PROMPT_2)
    preflight_rows.append({
        "id": row["id"],
        "aspect": aspect,
        "width": width,
        "height": height,
        **report,
        "positive_prompt": positive,
    })

preflight_df = pd.DataFrame(preflight_rows)
preflight_df

## Generate 3 Iteration Images

This default run creates **one image per prompt**.

Do not scale this notebook to a large batch until these three outputs satisfy the basic goals:

- one strong face/eyes read
- one strong full-body silhouette
- one strong hard-surface or environmental read

If one of those categories fails, fix prompting first.


In [ ]:
metadata = []
generated_paths = []

for p_idx, row in enumerate(tqdm(prompt_rows, desc="Prompts"), start=1):
    aspect = row.get("aspect", "square")
    width, height = ASPECT_SIZES.get(aspect, ASPECT_SIZES["square"])
    positive = build_positive_prompt(row["content_tags"])
    report = validate_prompt_channels(positive, PROMPT_2_STYLE, NEGATIVE_PROMPT, NEGATIVE_PROMPT_2)

    for v_idx in range(NUM_IMAGES_PER_PROMPT):
        seed = BASE_SEED + (p_idx * 1000) + v_idx
        image = call_pipe(
            prompt=positive,
            prompt_2=PROMPT_2_STYLE,
            negative_prompt=NEGATIVE_PROMPT,
            negative_prompt_2=NEGATIVE_PROMPT_2,
            width=width,
            height=height,
            seed=seed,
        )

        stem = f"{row['id']}_seed{seed}"
        image_path = IMAGES_DIR / f"{stem}.png"
        caption_path = IMAGES_DIR / f"{stem}.txt"

        image.save(image_path)
        caption_path.write_text(row["caption"].strip() + "
", encoding="utf-8")
        generated_paths.append(image_path)

        metadata.append({
            "id": row["id"],
            "seed": seed,
            "aspect": aspect,
            "width": width,
            "height": height,
            "image": str(image_path),
            "caption_file": str(caption_path),
            "caption": row["caption"],
            "content_tags": row["content_tags"],
            "prompt": positive,
            "prompt_2": PROMPT_2_STYLE,
            "negative_prompt": NEGATIVE_PROMPT,
            "negative_prompt_2": NEGATIVE_PROMPT_2,
            **report,
            "num_inference_steps": NUM_INFERENCE_STEPS,
            "guidance_scale": GUIDANCE_SCALE,
            "clip_skip": CLIP_SKIP,
            "generated_at_utc": datetime.utcnow().isoformat(),
        })

        display(image.resize((min(320, image.width), int(image.height * min(320, image.width) / image.width))))

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

metadata_df = pd.DataFrame(metadata)
metadata_df.to_csv(OUTPUT_ROOT / "metadata.csv", index=False)
metadata_df.to_json(OUTPUT_ROOT / "metadata.jsonl", orient="records", lines=True, force_ascii=False)
metadata_df

## Contact Sheet and Archive

The contact sheet is the fastest way to judge whether the run is worth iterating on.

Reject and revise if you see:

- sketch-like finish
- muddy or weak color
- poor face readability
- weak silhouette
- broken hands or anatomy
- generic anime drift
- missing material separation between skin, cloth, and metal


In [ ]:
contact_sheet_path = OUTPUT_ROOT / "contact_sheet_v4.jpg"
make_contact_sheet(generated_paths, contact_sheet_path, cols=3)

archive_path = OUTPUT_ROOT / "output_archive_v4.zip"
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if path.is_file() and path != archive_path:
            zf.write(path, path.relative_to(OUTPUT_ROOT))

print("Contact sheet:", contact_sheet_path)
print("Archive:", archive_path)
Image.open(contact_sheet_path)

## Next Iteration Rules

If the 3-image run still looks weak, do **not** jump straight to a larger dataset batch.

Adjust in this order:

1. Increase `GUIDANCE_SCALE` from `7.0` to `7.5`.
2. Strengthen `QUALITY_TAGS` with `vivid full color` or `saturated color` if color remains weak.
3. Tighten `content_tags` so the subject read is clearer.
4. Remove any tag that feels decorative but not causally useful.
5. Only after the 3-image pass works, raise `NUM_IMAGES_PER_PROMPT` and expand the prompt file.

The goal here is not volume. The goal is to find a prompt/control recipe that produces images worth curating into a real LoRA dataset.
